In [ ]:
!wget https://raw.githubusercontent.com/JoseCaliz/dotfiles/main/css/gruvbox.css 2>/dev/null 1>&2
!pip install feature_engine 2>/dev/null 1>&2
    
from IPython.core.display import HTML
with open('./gruvbox.css', 'r') as file:
    custom_css = file.read()

HTML(custom_css)

# Introduction

Happy New year! Welcome to a new notebook to tackle a new playground series competition. From this year onward, the competition is no longer tabular exclusive; it may contain other forms of data, more information in [this post](https://www.kaggle.com/discussions/general/375342#2081976) by @inversion. Nevertheless, a good EDA is a way to gain insights about the data and improve your models by applying feature engineering. This notebook attempts to provide a complete starter guide so others can build on top.

Thanks for reading. Don't forget to upvote if you find it useful.

* v2: Modify distribution plots to include scatter plot with the mean target per bin.
* v3: Changed MSE to RMSE to reflect the metric from the competition.
* v4: Added additional map with the target as the color. It shows that most expensive properties are located in big cities
* v5: Added LGBM `sample_weight` validation

# Library Import

Some library import and some configurations of matplotlib.

In [ ]:
from colorama import Fore, Style
from datetime import timedelta
from feature_engine.encoding import OneHotEncoder
from folium import plugins
from folium.plugins import HeatMap
from itertools import product as cartessian_product
from matplotlib.ticker import LinearLocator
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as mse
from sklearn.model_selection import KFold
from time import time

import branca
import folium
import matplotlib
import matplotlib as mpl
import matplotlib.cm as cmap
import matplotlib.colors as mpl_colors
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import lightgbm as lgbm
import numpy as np
import pandas as pd
import seaborn as sns
import scipy
import warnings

def hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16)/255 for i in (0, 2, 4))

palette = ['#b4d2b1', '#568f8b', '#1d4a60', '#cd7e59', '#ddb247', '#d15252']
palette_rgb = [hex_to_rgb(x) for x in palette]
cmap = mpl_colors.ListedColormap(palette_rgb)
colors = cmap.colors
bg_color= '#fdfcf6'

custom_params = {
    "axes.spines.right": False,
    "axes.spines.top": False,
    'grid.alpha':0.3,
    'figure.figsize': (16, 6),
    'axes.titlesize': 'Large',
    'axes.labelsize': 'Large',
    'figure.facecolor': bg_color,
    'axes.facecolor': bg_color
}

sns.set_theme(
    style='whitegrid',
    palette=sns.color_palette(palette),
    rc=custom_params
)

warnings.simplefilter("ignore", UserWarning)

# The Data

The dataset was generated using [California Housing Dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_california_housing.html). This a regression task.

Some key aspects are:

* Feature distributions are close to, but not exactly the same, as the original.
* We can use the original dataset as part of data augmentation.
* It is a fairly light dataset, so we can use lots of algorithms.

Columns description [taken from this link](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset):

- MedInc median income in block group
- HouseAge median house age in block group
- AveRooms average number of rooms per household
- AveBedrms average number of bedrooms per household
- Population block group population
- AveOccup average number of household members
- Latitude block group latitude
- Longitude block group longitude

In [ ]:
train_df = pd.read_csv('/kaggle/input/playground-series-s3e1/train.csv', index_col=0)
test_df = pd.read_csv('/kaggle/input/playground-series-s3e1/test.csv', index_col=0)
features = train_df.columns[:-1]

# load original dataset
original_df, original_y = fetch_california_housing(return_X_y=True)
original_df = pd.DataFrame(original_df, columns=features)
original_df['MedHouseVal'] = original_y

# EDA

## Data Size

**Insights**:
- This a fairly light dataset, we can use several models, ensembles, feature engineering and multiple tricks without lots of computational power.
- Generated dataset has more records that original dataset.

In [ ]:
print('Train shape:            ', train_df.shape)
print('Test shape:             ', test_df.shape)
print('Original Dataset shape: ', original_df.shape)

## Distributions

It's alwas important to check your data distribution, that will give you a sense of which variables behave oddly and may require preprocessing.
The following helper function will plot the mean value of the target column with the same scale of the generated histplots. It's a quick a fun way to visualize the relationships between the features and the target.

In [ ]:
def add_secondary_plot(
    df, column, target_column, ax, n_bins, color=3,
    show_yticks=False,
):
    secondary_ax = ax.twinx()
    bins = pd.cut(df[column], bins=n_bins)
    bins = pd.IntervalIndex(bins)
    bins = (bins.left + bins.right) / 2
    target = df.groupby(bins)[target_column].mean()
    target.plot(
        ax=secondary_ax, linestyle='',
        marker='.', color=colors[color], label=f'Mean {target_column}'
    )
    secondary_ax.grid(visible=False)
    
    if not show_yticks:
        secondary_ax.get_yaxis().set_ticks([])
        
    return secondary_ax

### Train Vs Test

**Insights**:
- There isn't notable differences between train and test, we probably do not need to do a in-depth adversarial validation.
- `AveOccup`, `AveBdrms`, `AveRooms`, `Population` have really skewed distributions, consider using `clip` function if there isn't a strong impact of outlier values. This may help especially for models that requires feature scaling.

In [ ]:
n_bins = 50
histplot_hyperparams = {
    'kde':True,
    'alpha':0.4,
    'stat':'percent',
    'bins':n_bins
}

columns = features
fig, ax = plt.subplots(2, 4, figsize=(16, 10))
ax = ax.flatten()

for i, column in enumerate(columns):
    plot_axes = [ax[i]]
    
    sns.histplot(
        train_df[column], label='Train',
        ax=ax[i], color=colors[0], **histplot_hyperparams
    )
    
    sns.histplot(
        test_df[column], label='Test',
        ax=ax[i], color=colors[1], **histplot_hyperparams
    )
    
    # Secondary axis to show mean of target
    ax2 = add_secondary_plot(train_df, column, 'MedHouseVal', ax[i], n_bins, color=4)
    
    # titles
    ax[i].set_title(f'{column} Distribution');
    ax[i].set_xlabel(None)
    
    # remove axes to show only one at the end
    plot_axes = [ax[i], ax2]
    handles = []
    labels = []
    for plot_ax in plot_axes:
        handles += plot_ax.get_legend_handles_labels()[0]
        labels += plot_ax.get_legend_handles_labels()[1]
        plot_ax.legend().remove()
    
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.04), fontsize=14, ncol=3)
plt.tight_layout()

### Train vs Original

**Insights**:
- Biggest difference between generated data and original data is on `HouseAge`
- We should check pairwise correlations or distribution to get a sense of how train and original dataset differ.
- There is an interesting case for `Population`, the correlation relationship is positive for train dataset but inverted for the original dataset. Blindly augmenting this dataset may not yield the best result

In [ ]:
n_bins = 50
histplot_hyperparams = {
    'kde':True,
    'alpha':0.4,
    'stat':'percent',
    'bins':n_bins
}

columns = features
fig, ax = plt.subplots(2, 4, figsize=(16, 10))
ax = ax.flatten()

for i, column in enumerate(columns):
    sns.histplot(
        train_df[column], label='Train',
        ax=ax[i], color=colors[0], **histplot_hyperparams
    )
    
    sns.histplot(
        original_df[column], label='Original',
        ax=ax[i], color=colors[1], **histplot_hyperparams
    )
    
    # Secondary axis to show mean of target
    ax2 = add_secondary_plot(train_df, column, 'MedHouseVal', ax[i], n_bins, color=4)
    ax3 = add_secondary_plot(original_df, column, 'MedHouseVal', ax[i], n_bins, color=5)
    
    # titles
    ax[i].set_title(f'{column} Distribution');
    ax[i].set_xlabel(None)
    
    # remove axes to show only one at the end
    plot_axes = [ax[i], ax2, ax3]
    handles = []
    labels = []
    for plot_ax in plot_axes:
        handles += plot_ax.get_legend_handles_labels()[0]
        labels += plot_ax.get_legend_handles_labels()[1]
        plot_ax.legend().remove()
    
labels = ['Train', 'Test', 'Train Mean MedHouseVal', 'Orginial Mean MedHouseVal']
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.04), fontsize=13, ncol=4)
plt.tight_layout()

## Missing Values

**Insights**
- There are no null values, this competition won't involve any null filling technique.

In [ ]:
fig = plt.figure(tight_layout=True, figsize=(20, 9))
gs = mpl.gridspec.GridSpec(2, 2)
null_table = []

# plot bars
for i, (k, df) in enumerate({'train':train_df, 'test':test_df}.items()):
    ax = fig.add_subplot(gs[i, 0])
    null_table.append((df.isnull().sum()/df.shape[0]).rename(k))
    
    sns.histplot(
        data=df.drop('MedHouseVal', errors='ignore', axis=1).isna().melt(value_name="missing"),
        y="variable",
        hue="missing",
        multiple="fill",
        ax=ax
    )

    ax.xaxis.set_major_locator(LinearLocator(21))
    ax.xaxis.set_major_formatter('{:.0%}'.format)
    ax.set_title(k, fontsize=15)
    ax.set_xlabel('Null Percentage');

# plot tables
ax = fig.add_subplot(gs[:, 1])
table_data = pd.concat(null_table, axis=1).applymap('{:.2%}'.format)

## colors
row_color, col_color = list(colors[0]), list(colors[1])

row_color.append(0.4)
col_color.append(0.4)

table_data.drop('MedHouseVal', inplace=True)
col_labels = table_data.columns.tolist()
row_labels = table_data.index.tolist()

ax.get_xaxis().set_visible(False)
ax.get_yaxis().set_visible(False)
ax.set_frame_on(None)

table = ax.table(
    cellText=table_data.to_numpy(),
    rowLabels=row_labels,
    colLabels=col_labels,
    loc='center',
    colWidths=[0.1, 0.1],
    rowColours=[row_color] * table_data.shape[0],
    colColours=[col_color] * table_data.shape[1],
);
table.set_fontsize(13)
table.scale(2, 3)

print('Original df null count\n', original_df.isnull().sum(), sep='')

## Correlations

**Insights**:
- Correlations from train and original datasets are different.
- Pairwise correlation between features and `MedHousVal` are similar between train and original dataset.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 20))
float_types = [np.float16, np.float32, np.float64]
float_columns = train_df.select_dtypes(include=float_types).columns
cbar_ax = fig.add_axes([.91, .39, .01, .2])

names = ['Train', 'Original']
for i, df in enumerate([train_df, original_df]):
    
    corr = df[float_columns].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))

    sns.heatmap(
        corr, mask=mask, cmap='inferno',
        vmax=0.8, vmin=-1,
        center=0, annot=False, fmt='.3f',
        square=True, linewidths=.5,
        ax=ax[i],
        cbar=False,
        cbar_ax=None
    );

    ax[i].set_title(f'Correlation matrix for {names[i]} df', fontsize=14)

df = test_df
float_columns = test_df.select_dtypes(include=float_types).columns
corr = test_df[float_columns].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr, mask=mask, cmap='inferno',
    vmax=0.8, vmin=-1,
    center=0, annot=False, fmt='.3f',
    square=True, linewidths=.5,
    cbar_kws={"shrink":.5, 'orientation':'vertical'},
    ax=ax[2],
    cbar=True,
    cbar_ax=cbar_ax
);
ax[2].set_title(f'Correlation matrix for Test', fontsize=14)
fig.tight_layout(rect=[0, 0, .9, 1]);

## Bivariate Distribution with Target

Even though correlations will give us a clear picture of linear relationships, it's also important to plot distribution to discover non relationships between variables and the target


### Train Dataset

**Insights**:
- Greater `MedInc` the greater the `MedHouseVal`, validated from correlations and from bivariate distribution, a similar effect can be seen on `AveRoooms` but is less evident.


In [ ]:
histplot_hyperparams = {
    'kde':True,
    'alpha':0.4,
    'stat':'percent'
}

columns = features
fig, ax = plt.subplots(2, 4, figsize=(16, 10), sharey=True)
ax = ax.flatten()

for i, column in enumerate(columns):
    sns.histplot(
        x=train_df[column],
        y=train_df['MedHouseVal'],
        label='Train',
        ax=ax[i], color=colors[0], **histplot_hyperparams
    )
    
    ax[i].set_title(f'{column} Distribution vs MedHouseVal', fontsize=10);
    ax[i].set_xlabel(None)
    ax[i].legend()

plt.tight_layout()

### Original dataset

**Insights**:
- Similar relationships as generated data but distribution shapes are different.
- `AveRooms` has lots of outliers.

In [ ]:
histplot_hyperparams = {
    'kde':True,
    'alpha':0.4,
    'stat':'percent'
}

columns = features
fig, ax = plt.subplots(2, 4, figsize=(16, 10), sharey=True)
ax = ax.flatten()

for i, column in enumerate(columns):
    sns.histplot(
        x=original_df[column],
        y=train_df['MedHouseVal'],
        label='Train',
        ax=ax[i], color=colors[0], **histplot_hyperparams
    )
    
    ax[i].set_title(f'{column} Distribution  vs MedHouseVal', fontsize=10);
    ax[i].set_xlabel(None)

plt.tight_layout()

## Latitude And Longitude

**Insights**
- Data is distributed across the entire california state
- We can consider using socieconomic external data to have additional insights about geographic location
- Lots of properties in Los Angeles
- Train and test follow the same distribution of latitud-longitude pairs. random KFold Is enough


In [ ]:
heat_map = folium.Map(train_df[['Latitude', 'Longitude']].mean(axis=0),
                    zoom_start = 6) 

train_df['Latitude'] = train_df['Latitude'].astype(float)
train_df['Longitude'] = train_df['Longitude'].astype(float)

lat_long_list = [[row['Latitude'],row['Longitude']] for index, row in train_df.iterrows()]
HeatMap(lat_long_list, radius=10).add_to(heat_map)
heat_map

Uncomment for completeness but plot from original datasets is similar to the generated dataset

In [ ]:
# heat_map = folium.Map(original_df[['Latitude', 'Longitude']].mean(axis=0),
#                     zoom_start = 6) 

# original_df['Latitude'] = original_df['Latitude'].astype(float)
# original_df['Longitude'] = original_df['Longitude'].astype(float)

# lat_long_list = [[row['Latitude'],row['Longitude']] for index, row in original_df.iterrows()]
# HeatMap(lat_long_list, radius=10).add_to(heat_map)
# heat_map

Uncomment for completeness but plot from test datasets is similar to the generated dataset

In [ ]:
# heat_map = folium.Map(test_df[['Latitude', 'Longitude']].mean(axis=0),
#                     zoom_start = 6) 

# test_df['Latitude'] = test_df['Latitude'].astype(float)
# test_df['Longitude'] = test_df['Longitude'].astype(float)

# HeatMap(lat_long_list, radius=10).add_to(heat_map)
# heat_map

## Latitude and Longitud relationship with Target

**Insights**
- Most expensive properties tend to be located nearby and in big citiest like San Francisco and Los Angeles and that are close to the beaches

In [ ]:
inferno_colors = [
    (0, 0, 4),
    (40, 11, 84),
    (101, 21, 110),
    (159, 42, 99),
    (212, 72, 66),
    (245, 125, 21),
    (250, 193, 39),
    (252, 255, 164)
]

map = folium.Map(train_df[['Latitude', 'Longitude']].mean(axis=0), zoom_start = 6)
lat = list(train_df.Latitude)
lon = list(train_df.Longitude)
populations = list(train_df.Population)
targets = list(train_df.MedHouseVal)

# define colormap using inferno colors and normalizing them according MedHouseVal
cmap = branca.colormap.LinearColormap(
    inferno_colors, vmin=min(targets), vmax=max(targets)
)

for loc, population, target in zip(zip(lat, lon), populations, targets):
    folium.Circle(
        location=loc,
        radius=population,
        fill=True,
            color=cmap(target),
        fill_opacity=0.2,
        weight=0
    ).add_to(map)

map.add_child(cmap)
display(map)

# Baseline

## Only With Train data

In [ ]:
cv = KFold(shuffle=True, random_state=2023)
train_df_ = train_df[features]
target = train_df['MedHouseVal']
oof_preds = pd.Series(0, index=train_df.index)
start = time()
tr_losses = []
vl_losses = []
models = []

for fold, (tr_ix, vl_ix) in enumerate(cv.split(train_df, target)):
    X_tr, y_tr = train_df_.iloc[tr_ix], target.iloc[tr_ix]
    X_vl, y_vl = train_df_.iloc[vl_ix], target.iloc[vl_ix]
    
    model = LinearRegression()
    start_fold = time()
    print('_'*50)
    print(f'Fold {fold} | {timedelta(seconds=int(time()-start))}')
    
    model.fit(X_tr, y_tr)
    
    oof_preds.iloc[vl_ix] = model.predict(X_vl)
    tr_losses.append(np.sqrt(mse(y_tr, model.predict(X_tr))))
    vl_losses.append(np.sqrt(mse(y_vl, model.predict(X_vl))))
    models.append(model)

    print(f'Val RMSE: {Fore.BLUE}{vl_losses[-1]}{Style.RESET_ALL}')

print()
print(f'Mean Val RMSE: {Fore.GREEN}{np.mean(vl_losses)}{Style.RESET_ALL}')
print(f'OOF RMSE:      {Fore.GREEN}{np.sqrt(mse(target, oof_preds))}{Style.RESET_ALL}')

## Train + original dataset augmentation

**Insights**
- OOF preds benefits of having agumented dataset 0.98918 vs 0.73074 with augmentation


In [ ]:
cv = KFold(shuffle=True, random_state=2023)
train_df_ = train_df[features]
target = train_df['MedHouseVal']
oof_preds = pd.Series(0, index=train_df.index)
start = time()
tr_losses = []
vl_losses = []
models = []

for fold, (tr_ix, vl_ix) in enumerate(cv.split(train_df, target)):
    X_tr, y_tr = train_df_.iloc[tr_ix], target.iloc[tr_ix]
    X_vl, y_vl = train_df_.iloc[vl_ix], target.iloc[vl_ix]
    
    X_tr = pd.concat([X_tr, original_df[features]], axis=0)
    y_tr = pd.concat([y_tr, original_df['MedHouseVal']])
    
    model = LinearRegression()
    start_fold = time()
    print('_'*50)
    print(f'Fold {fold} | {timedelta(seconds=int(time()-start))}')
    
    model.fit(X_tr, y_tr)
    
    oof_preds.iloc[vl_ix] = model.predict(X_vl)
    tr_losses.append(np.sqrt(mse(y_tr, model.predict(X_tr))))
    vl_losses.append(np.sqrt(mse(y_vl, model.predict(X_vl))))
    models.append(model)

    print(f'Val RMSE: {Fore.BLUE}{vl_losses[-1]}{Style.RESET_ALL}')

print()
print(f'Mean Val RMSE: {Fore.GREEN}{np.mean(vl_losses)}{Style.RESET_ALL}')
print(f'OOF RMSE:      {Fore.GREEN}{np.sqrt(mse(target, oof_preds))}{Style.RESET_ALL}')

## Sample Weight

**Inisights**
* Using only train (sample_weight for original dataset is 0) has an oof of 0.57046
* Same weight for original and train datasets has an oof RMSE of 0.56652
* Best `sample_weight` results in oof RMSE of 0.56560 (≈ 0.001 gain)

In [ ]:
%%time
cv = KFold(shuffle=True, random_state=2023)
train_df_ = train_df[features].copy()
original_df_ = original_df.copy()
results = {}
models = {}

for i, weight in enumerate(np.linspace(0, 1, 100)):
    train_df_['weight'] = 1
    original_df_['weight'] = weight

    target = train_df['MedHouseVal']
    oof_preds = pd.Series(0, index=train_df.index)
    start = time()
    tr_losses = []
    vl_losses = []
    models[weight] = []

    for fold, (tr_ix, vl_ix) in enumerate(cv.split(train_df, target)):
        X_tr, y_tr = train_df_.iloc[tr_ix], target.iloc[tr_ix]
        X_vl, y_vl = train_df_.iloc[vl_ix], target.iloc[vl_ix]

        X_tr = pd.concat([X_tr, original_df_.drop(columns=['MedHouseVal'])], axis=0)
        y_tr = pd.concat([y_tr, original_df_['MedHouseVal']])
        
        w = X_tr.pop('weight')
        X_vl.pop('weight')

        model = lgbm.LGBMRegressor()
        start_fold = time()

        model.fit(X_tr, y_tr, sample_weight=w)
        oof_preds.iloc[vl_ix] = model.predict(X_vl)
        tr_losses.append(np.sqrt(mse(y_tr, model.predict(X_tr))))
        vl_losses.append(np.sqrt(mse(y_vl, model.predict(X_vl))))
        models[weight].append(model)

    if i % 10 == 0:
        print(f'#### Weight {weight:.2f}')
        print(f'Mean Val RMSE: {Fore.GREEN}{np.mean(vl_losses)}{Style.RESET_ALL}')
        print(f'OOF RMSE:      {Fore.GREEN}{np.sqrt(mse(target, oof_preds))}{Style.RESET_ALL}')
        print()
        
    results[weight] = np.sqrt(mse(target, oof_preds))

In [ ]:
fig, ax = plt.subplots()
best_weight, min_rmse = min(results, key=results.get), min(results.values())
ax.plot(results.keys(), results.values())
ax.axhline(min_rmse, linestyle='--', color=colors[2], alpha=0.3)

y_min, y_max = ax.get_ylim()
ax.axvline(
    best_weight, ymax=(min_rmse -  y_min)/(y_max - y_min),
    linestyle='--', color=colors[2], alpha=0.3
)

ax.text(best_weight, min_rmse-0.0002, f'{best_weight:.4f}', color=colors[3], fontsize=13)

ax.set_title('OOF RMSE vs sample_weight');
ax.set_ylabel('OOF RMSE')
ax.set_xlabel('sample_weight')

print(f'Min OOF RMSE: {min_rmse:.4f} at sample_weight = {best_weight:.3f}')

In [ ]:
train_df_ = train_df.copy()
train_df_['pred'] = oof_preds
train_df_['residual'] = (train_df_.MedHouseVal - train_df_.pred).abs()

inferno_colors = [
    (0, 0, 4),
    (40, 11, 84),
    (101, 21, 110),
    (159, 42, 99),
    (212, 72, 66),
    (245, 125, 21),
    (250, 193, 39),
    (252, 255, 164)
]

map = folium.Map(train_df[['Latitude', 'Longitude']].mean(axis=0), zoom_start = 6, tiles='cartodbpositron')
lat = list(train_df_.Latitude)
lon = list(train_df_.Longitude)
populations = list(train_df_.Population)
targets = list(train_df_.residual)

# define colormap using inferno colors and normalizing them according MedHouseVal
cmap = branca.colormap.LinearColormap(
    inferno_colors, vmin=min(targets), vmax=max(targets)
)

for loc, population, target in zip(zip(lat, lon), populations, targets):
    folium.Circle(
        location=loc,
        radius=200,
        fill=True,
            color=cmap(target),
        fill_opacity=0.6,
        weight=0
    ).add_to(map)

map.add_child(cmap)
display(map)

## Generate test Predictions

Using a mean ensemble from each fold model

In [ ]:
best_models = models[best_weight]
preds = pd.Series(0, index=test_df.index)
for model in best_models:
    preds += model.predict(test_df)

preds /= len(best_models)
preds.to_csv('sumbission.csv')